In [1]:
import random
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Dataset
from transformers import AutoTokenizer
from collections import Counter

In [2]:
# ----------------------------------------
# GPT-2 tokenizer
# ----------------------------------------
tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [4]:
!wget http://mattmahoney.net/dc/text8.zip
!unzip text8.zip

--2026-06-10 07:07:59--  http://mattmahoney.net/dc/text8.zip
Resolving mattmahoney.net (mattmahoney.net)... 20.119.76.151
Connecting to mattmahoney.net (mattmahoney.net)|20.119.76.151|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 31344016 (30M) [application/zip]
Saving to: ‘text8.zip.1’

text8.zip.1         100%[===================>]  29.89M  12.2MB/s    in 2.4s    

2026-06-10 07:08:01 (12.2 MB/s) - ‘text8.zip.1’ saved [31344016/31344016]

Archive:  text8.zip
  inflating: text8                   


In [3]:
with open("text8", "r") as f:
    raw_text = f.read()

all_tokens = tokenizer.encode(raw_text)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (19974836 > 1024). Running this sequence through the model will result in indexing errors


In [4]:
counts = Counter(all_tokens)
total_tokens = len(all_tokens)

In [5]:
t = 1e-5  # try 1e-5 first, then 1e-4 if needed

def keep_token(token_id):
    freq = counts[token_id] / total_tokens
    prob_discard = 1 - np.sqrt(t / freq)
    return random.random() > prob_discard

In [6]:
filtered_tokens = [
    t for t in all_tokens if keep_token(t)
]

In [66]:
# ----------------------------------------
# Create skip-gram dataset
# target -> context
# ----------------------------------------
window_size = 8

class SkipGramDataset(Dataset):

    def __init__(self, tokens, window_size=5, num_samples=10_000_000):
        self.tokens = tokens
        self.window = window_size

        self.n = len(tokens)
        self.num_samples = num_samples

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        i = random.randint(0, self.n - 1)

        target = self.tokens[i]

        # sample context within window
        j = i + random.randint(-self.window, self.window)

        while j == i or j < 0 or j >= self.n:
            j = random.randint(0, self.n - 1)

        context = self.tokens[j]

        return target, context

In [28]:
# load dataset
batch_size = 4096

dataset = SkipGramDataset(filtered_tokens)

loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
)

In [7]:
# ----------------------------------------
# Embedding model
# ----------------------------------------
vocab_size = tokenizer.vocab_size
embedding_dim = 512

class SkipGramNegSampling(nn.Module):
    def __init__(self):
        super().__init__()

        self.target_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.context_embeddings = nn.Embedding(vocab_size, embedding_dim)

    def forward(self, target, context, negatives):
        """
        target:   (B,)
        context:  (B,)
        negatives:(B, K)
        """

        v = self.target_embeddings(target)     # (B, D)
        u_pos = self.context_embeddings(context)  # (B, D)
        u_neg = self.context_embeddings(negatives)  # (B, K, D)

        # positive score
        pos_score = torch.sum(v * u_pos, dim=1)   # (B,)
        pos_loss = F.logsigmoid(pos_score)

        # negative score
        neg_score = torch.bmm(
            u_neg, v.unsqueeze(2)
        ).squeeze(2)  # (B, K)

        neg_loss = F.logsigmoid(-neg_score).sum(1)

        loss = -(pos_loss + neg_loss).mean()

        return loss


word2vec_model = SkipGramNegSampling()

optimizer = optim.Adam(word2vec_model.parameters(), lr=1e-3)

In [8]:
device = torch.device("cuda")

In [9]:
word2vec_model = word2vec_model.to(device)

In [10]:
next(word2vec_model.parameters()).device

device(type='cuda', index=0)

In [11]:
import os
start_epoch = 0

if os.path.exists('checkpoint.pt'):
    checkpoint = torch.load(
        "checkpoint.pt",
        map_location=device
    )
    word2vec_model.load_state_dict(
        checkpoint["model_state_dict"]
    )
    
    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )
    
    start_epoch = checkpoint["epoch"] + 1

In [34]:
vocab_size = len(tokenizer)
freqs = np.zeros(vocab_size)

for t in all_tokens:
    freqs[t] += 1

freqs = freqs ** 0.75
freqs = freqs / freqs.sum()

In [35]:
def sample_negatives(batch_size, K):
    return np.random.choice(
        vocab_size,
        size=(batch_size, K),
        p=freqs
    )

In [73]:
# ----------------------------------------
# Training
# ----------------------------------------
epochs = 15

K = 40  # number of negatives

for epoch in range(start_epoch, epochs):

    total_loss = 0

    for i, (target_batch, context_batch) in enumerate(loader):

        negatives = torch.tensor(
            sample_negatives(target_batch.size(0), K),
            device=device,
            dtype=torch.long
        )

        target_batch = target_batch.to(device)
        context_batch = context_batch.to(device)
        negatives = negatives.to(device)

        optimizer.zero_grad()

        loss = model(target_batch, context_batch, negatives)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # -------------------------
        # progress logging
        # -------------------------
        if i % 100 == 0:
            print(f"epoch {epoch} | step {i} | loss {loss.item():.4f}")

    print(
        f"Epoch {epoch + 1} finished | "
        f"avg loss = {total_loss / len(loader):.4f}"
    )

    print("Saving checkpoint...")
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }, "checkpoint.pt")

epoch 10 | step 0 | loss 6.8159
epoch 10 | step 100 | loss 6.7605
epoch 10 | step 200 | loss 6.6540
epoch 10 | step 300 | loss 6.7698
epoch 10 | step 400 | loss 6.6367
epoch 10 | step 500 | loss 6.6060
epoch 10 | step 600 | loss 6.4812
epoch 10 | step 700 | loss 6.4838
epoch 10 | step 800 | loss 6.4358
epoch 10 | step 900 | loss 6.5340
epoch 10 | step 1000 | loss 6.3494
epoch 10 | step 1100 | loss 6.4493
epoch 10 | step 1200 | loss 6.3332
epoch 10 | step 1300 | loss 6.3933
epoch 10 | step 1400 | loss 6.2911
epoch 10 | step 1500 | loss 6.3047
epoch 10 | step 1600 | loss 6.3599
epoch 10 | step 1700 | loss 6.2407
epoch 10 | step 1800 | loss 6.2844
epoch 10 | step 1900 | loss 6.2843
epoch 10 | step 2000 | loss 6.2096
epoch 10 | step 2100 | loss 6.2024
epoch 10 | step 2200 | loss 5.9481
epoch 10 | step 2300 | loss 6.1044
epoch 10 | step 2400 | loss 6.0558
Epoch 11 finished | avg loss = 6.3719
Saving checkpoint...
epoch 11 | step 0 | loss 6.1130
epoch 11 | step 100 | loss 6.0351
epoch 11 | s

In [12]:
def nearest_tokens(model, tokenizer, query, top_k=20):

    word2vec_model.eval()

    with torch.no_grad():

        token_ids = tokenizer.encode(query)

        embeddings = word2vec_model.target_embeddings.weight.detach()

        embeddings = F.normalize(embeddings, dim=1)

        query_vector = embeddings[token_ids].mean(dim=0)

        similarities = embeddings @ query_vector

        # Exclude the query tokens themselves
        for token_id in token_ids:
            similarities[token_id] = -1

        top_ids = similarities.topk(top_k).indices

        print()
        print("Query:", [tokenizer.decode([token]) for token in token_ids])
        print()

        for idx in top_ids:

            idx = idx.item()

            print(
                f"{idx:6d}",
                f"{similarities[idx]:.4f}",
                repr(tokenizer.decode([idx]))
            )

In [13]:
nearest_tokens(word2vec_model, tokenizer, " government")
nearest_tokens(word2vec_model, tokenizer, " computer")
nearest_tokens(word2vec_model, tokenizer, " river")


Query: [' government']

  2151 0.7276 ' party'
  1964 0.7249 ' political'
   739 0.7234 ' under'
  2585 0.7141 ' states'
  1181 0.7084 ' state'
   262 0.6990 ' the'
 16503 0.6966 ' united'
   286 0.6933 ' of'
   625 0.6856 ' over'
  2260 0.6839 ' national'
   416 0.6804 ' by'
   290 0.6801 ' and'
  1866 0.6778 ' members'
  1175 0.6769 ' war'
   284 0.6767 ' to'
   287 0.6763 ' in'
   663 0.6751 ' its'
  1893 0.6744 ' president'
  1028 0.6741 ' against'
  1104 0.6697 ' support'

Query: [' computer']

  3341 0.6898 ' systems'
  1080 0.6566 ' system'
   973 0.6464 ' used'
   779 0.6391 ' use'
  3788 0.6302 ' software'
   329 0.6207 ' for'
   884 0.6135 ' such'
  1479 0.6053 ' free'
   766 0.6049 ' see'
   584 0.6016 ' other'
   670 0.5997 ' work'
   351 0.5969 ' with'
  2656 0.5966 ' original'
  1430 0.5964 ' program'
  1321 0.5958 ' information'
   290 0.5957 ' and'
  1912 0.5940 ' based'
   257 0.5908 ' a'
   281 0.5857 ' an'
   635 0.5831 ' also'

Query: [' river']

  5093 0.6340 ' no

In [ ]:
# ----------------------------------------
# Building an MLP
# ----------------------------------------

In [14]:
class NextTokenDataset(Dataset):

    def __init__(self, tokens, context_size=8):
        self.tokens = tokens
        self.context_size = context_size

    def __len__(self):
        return len(self.tokens) - self.context_size

    def __getitem__(self, idx):

        x = self.tokens[idx:idx+self.context_size]
        y = self.tokens[idx+self.context_size]

        return (
            torch.tensor(x, dtype=torch.long),
            torch.tensor(y, dtype=torch.long)
        )

In [15]:
dataset = NextTokenDataset(filtered_tokens, context_size=8)

In [16]:
class MLPLanguageModel(nn.Module):

    def __init__(
        self,
        pretrained_embeddings,
        hidden_dim=1024
    ):
        super().__init__()

        vocab_size, embedding_dim = pretrained_embeddings.shape

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        # initialize from Word2Vec
        self.embedding.weight.data.copy_(pretrained_embeddings)

        self.norm = nn.LayerNorm(embedding_dim)

        context_size = 8

        self.fc1 = nn.Linear(
            context_size * embedding_dim,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            vocab_size
        )

    def forward(self, x):

        # x: (batch_size, context_size)

        x = self.embedding(x)

        x = self.norm(x)    # normalize per token

        # (B, context_size, emb_dim)
        x = x.reshape(x.size(0), -1)

        x = F.relu(self.fc1(x))

        logits = self.fc2(x)

        return logits

In [17]:
pretrained_embeddings = (
    word2vec_model.target_embeddings.weight.detach().clone()
)

model = MLPLanguageModel(
    pretrained_embeddings,
    hidden_dim=1024
).to(device)

In [18]:
loader = DataLoader(
    dataset,
    batch_size=4096,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

In [19]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

In [25]:
start_epoch = 0

if os.path.exists('mlp_norm.pt'):
    mlp_checkpoint = torch.load(
        "mlp_norm.pt",
        map_location=device
    )
    model.load_state_dict(
        mlp_checkpoint["model_state_dict"]
    )
    
    optimizer.load_state_dict(
        mlp_checkpoint["optimizer_state_dict"]
    )
    
    start_epoch = mlp_checkpoint["epoch"] + 1

In [27]:
epochs = 30

for epoch in range(start_epoch, epochs):

    total_loss = 0

    for i, (x, y) in enumerate(loader):

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        if i % 100 == 0:
            print(
                f"epoch {epoch} "
                f"step {i} "
                f"loss {loss.item():.4f}"
            )

    print(
        f"Epoch {epoch} "
        f"avg loss {total_loss / len(loader):.4f}"
    )
    print("Saving checkpoint...")
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }, "mlp_norm.pt")

epoch 20 step 0 loss 7.6134
epoch 20 step 100 loss 7.5818
epoch 20 step 200 loss 7.5877
epoch 20 step 300 loss 7.5449
epoch 20 step 400 loss 7.7196
epoch 20 step 500 loss 7.7043
epoch 20 step 600 loss 7.7438
epoch 20 step 700 loss 7.7561
epoch 20 step 800 loss 7.7925
epoch 20 step 900 loss 7.7238
epoch 20 step 1000 loss 7.7431
epoch 20 step 1100 loss 7.7872
epoch 20 step 1200 loss 7.7553
Epoch 20 avg loss 7.6958
Saving checkpoint...
epoch 21 step 0 loss 7.3068
epoch 21 step 100 loss 7.3075
epoch 21 step 200 loss 7.3248
epoch 21 step 300 loss 7.3126
epoch 21 step 400 loss 7.3784
epoch 21 step 500 loss 7.4454
epoch 21 step 600 loss 7.4744
epoch 21 step 700 loss 7.4994
epoch 21 step 800 loss 7.5574
epoch 21 step 900 loss 7.5156
epoch 21 step 1000 loss 7.5352
epoch 21 step 1100 loss 7.6342
epoch 21 step 1200 loss 7.5861
Epoch 21 avg loss 7.4628
Saving checkpoint...
epoch 22 step 0 loss 7.0577
epoch 22 step 100 loss 7.1623
epoch 22 step 200 loss 7.1866
epoch 22 step 300 loss 7.1575
epoch 22

In [28]:
def predict_next(model, tokenizer, text):

    model.eval()

    ids = tokenizer.encode(text)

    ids = ids[-8:]

    while len(ids) < 8:
        ids = [0] + ids

    x = torch.tensor([ids], device=device)

    with torch.no_grad():

        logits = model(x)

        #next_token = logits.argmax(dim=-1)
        T = 0.7
        probs = torch.softmax(logits / T, dim=-1)

        top_ids = probs[0].topk(20).indices
        
        for idx in top_ids:
            print(
                tokenizer.decode([idx.item()]),
                probs[0, idx].item()
            )
        
        next_token = torch.multinomial(
            probs[0],
            num_samples=1
        )

    return tokenizer.decode(next_token.tolist())

In [29]:
predict_next(model, tokenizer, "the united states of")

 three 0.6633735299110413
 m 0.09551335126161575
 of 0.08674292266368866
 zero 0.03149527683854103
 the 0.026845883578062057
 players 0.020556068047881126
 one 0.011104405857622623
 and 0.010708781890571117
 to 0.010582203976809978
 a 0.010542717762291431
 y 0.005609735380858183
 seven 0.005208820570260286
an 0.005158396903425455
 an 0.002645942149683833
 k 0.0023628578055649996
 nine 0.0022271207999438047
 for 0.0016044228104874492
 five 0.0010348899522796273
 l 0.0009335445356555283
 use 0.0008601724402979016


' the'